# Colab GPU — chỉ chạy full: neurovoz2024 + sweep

15/16 model đã có số **full-protocol** (chạy trên CPU). Notebook này chỉ chạy nốt 2 thứ còn rút gọn:
**`neurovoz2024`** (mel-CNN from-scratch, LOSO 22-fold × 3 seed × 40 epoch) và **`sweep`** (toàn bộ 1091 câu),
rồi cập nhật lại stats/figures/report trên toàn bộ 16 model.

**Chuẩn bị (làm 1 lần):**
1. Nén thư mục dự án `Final_ARS_2/` **bao gồm cả `artifacts/`** (để giữ kết quả full của 15 model + per-utterance preds).
   - Có thể loại `artifacts/cache/` và `artifacts/preprocessed/` cho nhẹ — notebook sẽ tự trích lại `melspec` nếu thiếu.
   - **Bắt buộc giữ `artifacts/results/`** (chứa `model_comparison.csv`, `per_utterance/`).
2. Upload `Final_ARS_2.zip` lên Google Drive (vd `MyDrive/Final_ARS_2.zip`).
3. Runtime → Change runtime type → **GPU** (T4 đủ). Chạy lần lượt các cell.

In [ ]:
# 1) Kiểm tra GPU
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Hãy bật GPU: Runtime > Change runtime type > GPU'

In [ ]:
# 2) Mount Drive và giải nén dự án (GIỮ NGUYÊN artifacts đã có -> không xoá!)
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/Final_ARS_2.zip'   # <-- sửa cho đúng đường dẫn zip của bạn
import zipfile, os, glob
os.makedirs('/content/proj', exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/proj')
root = os.path.dirname(glob.glob('/content/proj/**/run.py', recursive=True)[0])
os.chdir(root)
print('project root:', root)
# xác nhận kết quả 15 model cũ còn nguyên
import pandas as pd
mc = 'artifacts/results/model_comparison.csv'
assert os.path.exists(mc), 'THIẾU artifacts/results/model_comparison.csv -> hãy nén kèm artifacts/results/'
print('model_comparison hiện có', len(pd.read_csv(mc)), 'model')

In [ ]:
# 3) Cài dependency (KHÔNG cài lại torch — Colab đã có bản GPU)
!pip -q install soundfile librosa soxr praat-parselmouth transformers xgboost pyloudnorm thop psutil pyyaml tqdm tabulate

In [ ]:
# 4) Đảm bảo có manifest + melspec cache (neurovoz & sweep cần). Không đụng cache khác.
import os, pandas as pd
from src.utils.common import load_config, resolve
from src.features import extract, cache
cfg = load_config()

if not os.path.exists(resolve(cfg['paths']['manifest'])):
    from src.data import build_manifest; build_manifest.build(cfg)
df = pd.read_csv(resolve(cfg['paths']['manifest']))
print('manifest:', len(df), 'utterances')

key = cache.cache_key('melspec', cache.feature_spec(cfg))
if cache.exists(cfg, 'melspec', key):
    print('melspec cache đã có [%s]' % key)
else:
    print('melspec chưa có -> trích (vài phút)...')
    extract.build_light(cfg, df, ['melspec'])

In [ ]:
# 5) Chạy RIÊNG neurovoz2024 ở FULL protocol (LOSO 22-fold, 3 seed, 40 epoch) — GPU
#    (ghi đè dòng neurovoz rút gọn trong model_comparison.csv)
!python run.py deep --only neurovoz2024

In [ ]:
# 6) Sweep FULL (toàn bộ 1091 câu, k=5, 15 epoch) — GPU
!python run.py sweep

In [ ]:
# 7) Cập nhật lại thống kê + hình + báo cáo trên TOÀN BỘ 16 model (đã gồm neurovoz full)
!python run.py stats
!python run.py figures
!python run.py report

In [ ]:
# 8) Xem bảng 16 model (kiểm tra neurovoz đã full) + lưu kết quả về Drive
import pandas as pd, shutil
d = pd.read_csv('artifacts/results/model_comparison.csv').sort_values('auc_mean', ascending=False)
display(d[['model','feature_group','accuracy_mean','auc_mean','f1_mean','subj_accuracy_mean']].round(3))
print('\nneurovoz2024 row:'); display(d[d.model == 'neurovoz2024'])
print('\nsweep:'); display(pd.read_csv('artifacts/results/sweep_results.csv'))
shutil.make_archive('/content/drive/MyDrive/Final_ARS_2_artifacts', 'zip', 'artifacts')
print('Đã lưu: MyDrive/Final_ARS_2_artifacts.zip')

**Lưu ý:**
- Notebook này CHỈ chạy lại `neurovoz2024` + `sweep` (2 thứ còn rút gọn trên CPU); 15 model khác giữ nguyên số full từ `artifacts/results/`.
- Nếu muốn chạy lại **toàn bộ 16 model** từ đầu trên GPU: xoá `artifacts/` rồi `!python run.py all`.
- Kết quả cuối: `artifacts/results/*.csv`, hình `artifacts/figures/`, báo cáo `docs/RESULTS.md`.